In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
from fastai.vision.all import *
import torch

In [5]:
path=untar_data(URLs.MNIST)

<div><progress max="15683414" value="15687680"></progress> 100.03% [15687680/15683414 00:02&lt;00:00]</div>

In [6]:
train_paths={i:(path/'training'/str(i)).ls() for i in range(10)}
valid_paths={i:(path/'testing'/str(i)).ls() for i in range(10)}

In [7]:
train_tensors={i:[tensor(Image.open(j)) for j in images] for i,images in train_paths.items()}
valid_tensors={i:[tensor(Image.open(j)) for j in images] for i,images in valid_paths.items()}

In [8]:
train_stacked={i:torch.stack(tensors).float()/255 for i,tensors in train_tensors.items()}
valid_stacked={i:torch.stack(tensors).float()/255 for i,tensors in valid_tensors.items()}

In [11]:
train_x=torch.cat([train_stacked[i] for i in range(10)]).view(-1,28*28)
train_y=tensor([[1 if i==num else 0 for i in range(10)] for num, images in train_stacked.items() for _ in range(len(images))])

In [12]:
valid_x=torch.cat([valid_stacked[i] for i in range(10)]).view(-1,28*28)
valid_y=tensor([[1 if i==num else 0 for i in range(10)] for num, images in valid_stacked.items() for _ in range(len(images))])

In [14]:
dset=list(zip(train_x,train_y))
valid_dset=list(zip(valid_x,valid_y))

In [15]:
dl=DataLoader(dset,batch_size=256,shuffle=True)
valid_dl=DataLoader(valid_dset,batch_size=256,shuffle=True)

In [16]:
dls=DataLoaders(dl,valid_dl)

In [17]:
mean_numbers={i:stack.mean(0) for i,stack in train_stacked.items()}

In [18]:
def diff(a,b):
    return (a-b).abs().mean((-1,-2))

In [19]:
accuracy=[]
for i in range(10):
    errs=[diff(valid_stacked[i],mean_numbers[num]) for num in range(10)] #transformed one stack into 10 lists of numbers
    stacked_errs=torch.stack(errs) #tensor of 10 lists along 0 axis
    min=stacked_errs.min(0).indices
    accs=(tensor(i)==min).float().mean()
    accuracy.append(accs)
mean_accuracy=tensor(accuracy).float().mean()
mean_accuracy

tensor(0.6610)

In [20]:
def init_params(size,std=1.0): return torch.randn(size)*std
weights=init_params((28*28,10)).requires_grad_()
bias=init_params(10).requires_grad_()

In [21]:
#def mnist_loss(predictions, targets):
#    loss = F.cross_entropy(predictions, targets.squeeze())
    #return loss

In [22]:
def mnist_loss(predictions, targets):
    loss = F.cross_entropy(predictions, targets.float())
    return loss

In [23]:
lr=0.1

In [27]:
def batch_accuracy(xb,yb):
    correct=xb.argmax(1)==yb.argmax(1)
    return correct.float().mean()

In [28]:
for epoch in range(20):
    epoch_loss = 0
    accuracy=0
    count=0
    for x,y in dl:
        predictions=x@weights+bias
        loss=mnist_loss(predictions,y.float())
        loss.backward()
        with torch.no_grad():
            weights -= lr*weights.grad.data
            bias -= lr*bias.grad.data
        weights.grad.zero_()
        bias.grad.zero_()
        epoch_loss += mnist_loss(predictions,y.float()).item()
        count += 1
        accuracy += batch_accuracy(predictions,y)
    print(f'epoch {epoch} loss:{epoch_loss/count}, accuracy:{accuracy/count}')

epoch 0 loss:4.232016065780153, accuracy:0.42175307869911194
epoch 1 loss:1.6311429335715923, accuracy:0.6863974332809448
epoch 2 loss:1.242508108311511, accuracy:0.7528645396232605
epoch 3 loss:1.0654812924405361, accuracy:0.7840480804443359
epoch 4 loss:0.9571628174883254, accuracy:0.8029642701148987
epoch 5 loss:0.8843547293480406, accuracy:0.8158798813819885
epoch 6 loss:0.8284658315333914, accuracy:0.8261192440986633
epoch 7 loss:0.7844834325161386, accuracy:0.8338043093681335
epoch 8 loss:0.7486873255131092, accuracy:0.839932382106781
epoch 9 loss:0.7190904607164099, accuracy:0.8456948399543762
epoch 10 loss:0.6927890568337542, accuracy:0.8497561812400818
epoch 11 loss:0.6701373953768548, accuracy:0.8535793423652649
epoch 12 loss:0.6516562682517031, accuracy:0.8569592237472534
epoch 13 loss:0.6338768789108763, accuracy:0.8597185611724854
epoch 14 loss:0.618293291457156, accuracy:0.8619459271430969
epoch 15 loss:0.6029885638267436, accuracy:0.8637577891349792
epoch 16 loss:0.59006

In [29]:
learn=Learner(dls,opt_func=SGD,model=nn.Linear(28*28,10),metrics=batch_accuracy,loss_func=mnist_loss)

In [30]:
learn.fit(20,0.1)

epoch,train_loss,valid_loss,batch_accuracy,time
0,0.522318,0.448082,0.887300,00:00
1,0.404513,0.380144,0.897900,00:00
2,0.379020,0.351047,0.905400,00:00
3,0.353143,0.335409,0.908000,00:00
4,0.334018,0.324030,0.910600,00:00
5,0.336204,0.318556,0.912600,00:00
6,0.323216,0.311120,0.915100,00:00
7,0.320532,0.305443,0.917000,00:00
8,0.320735,0.302235,0.917200,00:00
9,0.315179,0.297967,0.917700,00:00


In [31]:
simple_net=nn.Sequential(
    nn.Linear(28*28,90),
    nn.ReLU(),
    nn.Linear(90,10))

In [32]:
learn_net=Learner(dls,opt_func=SGD,model=simple_net,metrics=batch_accuracy,loss_func=mnist_loss)

In [33]:
learn_net.fit(5,0.1)

epoch,train_loss,valid_loss,batch_accuracy,time
0,0.474672,0.380013,0.895800,00:00
1,0.337365,0.308095,0.912600,00:00
2,0.298374,0.281742,0.922400,00:00
3,0.264231,0.254783,0.928000,00:01
4,0.243097,0.230869,0.934200,00:00


In [34]:
dls_v=ImageDataLoaders.from_folder(path,train='training',valid='testing')

In [35]:
learn_v=vision_learner(dls_v,resnet18,pretrained=False)

In [36]:
learn_v.fit(1,0.1)

epoch,train_loss,valid_loss,time
0,1.247610,1.693116,06:07
